# ETL da tabela de alimentos — FittNutri

Este notebook é um **passeio guiado** pelo mesmo tratamento do `tratamento.py`. Toda a lógica fica no `.py`
(o notebook só chama e mostra), então os dois nunca ficam diferentes.

**O que o ETL faz**
1. Lê a TACO (597 alimentos) e o IBGE (1.971 alimentos).
2. Limpa os valores e deixa os nutrientes **por 1 g** (padrão do banco).
3. Descobre quantos gramas tem 1 colher, 1 xícara, 1 unidade... de cada alimento (medidas caseiras).
4. Marca de onde veio cada medida: **dado real** ou **valor aproximado**.
5. Gera o script SQL e, se você quiser, carrega no MySQL.

> Abra este notebook a partir da pasta `dados/ETL` (os arquivos são lidos pelo caminho relativo).

In [ ]:
import pandas as pd
import tratamento as t

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## 1. Lendo a TACO
A planilha tem o cabeçalho repetido a cada página e os grupos (Cereais, Frutas...) em linhas soltas. O ETL separa tudo isso.

In [ ]:
df_taco_bruto = t.extrair_taco(t.CAMINHO_TACO)
print(len(df_taco_bruto), "alimentos da TACO em", df_taco_bruto["grupo"].nunique(), "grupos")
df_taco_bruto[["grupo", "descricao_alimentos", "energia (kcal)", "proteina (g)"]].head()

## 2. Lendo o IBGE (POF 2008-2009)
O nome é montado como "Descrição - Preparação", por exemplo `Frango inteiro - Cozido(a)`.

In [ ]:
df_ibge_bruto = t.extrair_ibge(t.CAMINHO_IBGE)
print(len(df_ibge_bruto), "alimentos/preparações do IBGE")
t.transformar_ibge(df_ibge_bruto)[["nome", "energia_kcal", "proteina"]].head()

## 3. Tabela de medidas caseiras do IBGE
Uma linha por alimento + preparação, com os gramas de cada medida que o front usa (unidade, colher de sopa, colher de chá, xícara, ml).

In [ ]:
df_medidas = t.extrair_medidas(t.CAMINHO_MEDIDAS)
print(len(df_medidas), "alimentos/preparações do IBGE com alguma medida")
df_medidas.head()

## 4. Rodando o tratamento completo

`processar()` aplica as camadas, da mais confiável para a menos confiável. Cada camada só preenche o que as anteriores deixaram vazio:

| Origem | O que é | Confiança |
|---|---|---|
| `IBGE` | medida real do próprio alimento | alta |
| `IBGE_REF_TACO` | medida real que o IBGE copiou para um alimento da TACO | alta |
| `MAPEADO` | medida real ligada à mão a um alimento da TACO (`mapa_taco_ibge.csv`) | boa |
| `REFERENCIA_EXTERNA` | valor aproximado de uma tabela brasileira (`medidas_complementares.csv`) | **aproximada** |
| `ESTIMADO` | valor aproximado calculado pelo ETL (volume de líquidos ou mediana do grupo) | **aproximada** |

Uma colher varia muito de pessoa para pessoa. **Os valores aproximados dão só uma ordem de grandeza.**

In [ ]:
r = t.processar()
df = r["alimentos"]
print(r["qtd_taco"], "alimentos da TACO +", r["qtd_ibge"], "do IBGE =", len(df))
print(len(r["mapa"]), "ligações no mapa TACO -> IBGE;", len(r["complementares"]), "linhas em medidas_complementares.csv")

## 5. O que foi corrigido, descartado ou merece atenção

In [ ]:
print("Negativos de arredondamento corrigidos para 0:")
for fonte, nome, coluna, valor in r["corrigidos"]:
    print(f"  [{fonte}] {nome}: {coluna} = {valor:.3f}")

print()
print("Medidas descartadas por passarem do limite (provável erro de digitação):")
for fonte, nome, medida, valor in r["descartadas"]:
    print(f"  [{fonte}] {nome}: {medida} = {valor:g} g")

print()
print("Alertas de qualidade (não impedem a carga):")
for alerta in r["alertas"]:
    print("  -", alerta)

## 6. Estimativas por grupo
Só entram grupos com pelo menos 5 medidas reais e pouca variação. Grupos que misturam coisas diferentes (açúcares, laticínios, pratos prontos) ficam **sem** estimativa, porque o valor seria enganoso.

In [ ]:
tabela = pd.DataFrame(
    [(g, m.replace("gramas_", ""), med, n, disp) for (g, m), (med, n, disp) in sorted(r["medianas"].items())],
    columns=["grupo", "medida", "mediana (g)", "n", "dispersão"],
)
tabela

## 7. De onde veio cada medida
`medidas_fonte` guarda a origem **menos confiável** entre as medidas do alimento.

In [ ]:
resumo = df.groupby("fonte")["medidas_fonte"].value_counts(dropna=False).unstack(0).fillna(0).astype(int)
resumo.index = resumo.index.fillna("(sem nenhuma medida)")
display(resumo)
print("Alimentos com cada medida:")
df.groupby("fonte")[t.COLUNAS_MEDIDAS].count().T

## 8. Exemplos de cálculo
Valor da porção = valor por 1 g × gramas da porção.

In [ ]:
print(t.calcular_porcao(df, "Arroz, integral, cozido", 60)[["energia_kcal", "proteina", "carboidrato"]])
print()
print(t.calcular_medida(df, "Arroz integral", "colher_sopa", 3)[["energia_kcal", "proteina", "carboidrato"]])

## 9. Gravando os arquivos
O CSV e o relatório de medidas são só para conferência (o Git ignora). O script `.sql` é o que se roda no MySQL Workbench.

In [ ]:
GRAVAR_ARQUIVOS = True

colunas_novas, comandos = t.montar_comandos_sql(df)
if GRAVAR_ARQUIVOS:
    df.to_csv(t.CAMINHO_SAIDA, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    t.salvar_relatorio(df, t.CAMINHO_RELATORIO)
    t.salvar_sql(colunas_novas, comandos, t.CAMINHO_SQL)
    print("Arquivos gravados em", t.PASTA)
print(len(comandos), "comandos SQL montados")

## 10. Carregando no MySQL (opcional)
Antes: suba o backend uma vez para ele criar a tabela `alimentos`. A senha vem da variável de ambiente `DB_PASSWORD`
(ou é pedida na hora). **Nunca escreva a senha neste arquivo.** Pode rodar de novo sem duplicar nada.

In [ ]:
CARREGAR_NO_MYSQL = False

if CARREGAR_NO_MYSQL:
    t.aplicar_no_mysql(colunas_novas, comandos)
else:
    print("Carga desligada. Troque CARREGAR_NO_MYSQL para True para carregar no MySQL.")